In [1]:
import pandas as pd

rawData = pd.read_csv('Urban_Air_Quality_and_Health_Impact.csv')

rawData.info()
rawData.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 46 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   datetime           1000 non-null   object 
 1   datetimeEpoch      1000 non-null   float64
 2   tempmax            1000 non-null   float64
 3   tempmin            1000 non-null   float64
 4   temp               1000 non-null   float64
 5   feelslikemax       1000 non-null   float64
 6   feelslikemin       1000 non-null   float64
 7   feelslike          1000 non-null   float64
 8   dew                1000 non-null   float64
 9   humidity           1000 non-null   float64
 10  precip             1000 non-null   float64
 11  precipprob         1000 non-null   float64
 12  precipcover        1000 non-null   float64
 13  preciptype         378 non-null    object 
 14  snow               1000 non-null   float64
 15  snowdepth          929 non-null    float64
 16  windgust           1000 n

(1000, 46)

# 📊 城市空氣品質相關特徵分析

本段分析目標：找出資料集中與「空氣品質 / 可能影響空氣品質」最相關的特徵。

## 分析步驟
1. 載入資料集並查看基本資訊 (資料列數、欄位、型別、缺失值)。
2. 自動偵測可能代表空氣品質的目標欄位 (關鍵字: `aqi`, `air`, `quality`, `pm`)，若無則以 `Health_Risk_Score` 或 `Severity_Score` 作為替代。
3. 數值特徵：計算 Pearson 與 Spearman 相關係數，排序顯示前幾名。
4. 類別特徵：使用相關比率 (Correlation Ratio, η) 評估與目標的關聯程度。
5. 彙整結果並提出可能的影響因子解讀。

> 備註：若使用健康或風險分數作為目標，結果代表潛在環境或氣象影響因子，而非直接污染物濃度。

In [2]:
# 列出 city 特徵的所有不重複值
unique_cities = rawData['City'].unique()
print(f"不重複的城市數量: {len(unique_cities)}")
print(f"\n所有城市:")
print(unique_cities)

不重複的城市數量: 10

所有城市:
['Phoenix' 'San Jose' 'San Antonio' 'Los Angeles' 'San Diego'
 'New York City' 'Chicago' 'Philadelphia' 'Dallas' 'Houston']


In [3]:
# 資料概覽與缺失值統計
cols = rawData.columns.tolist()
print(f"資料筆數: {len(rawData)}")
print(f"欄位數: {len(cols)}")
print("前 5 筆資料:")
display(rawData.head())
print("欄位型別:")
print(rawData.dtypes)
print("缺失值統計 (非零者顯示):")
missing = rawData.isna().sum()
print(missing[missing>0])

資料筆數: 1000
欄位數: 46
前 5 筆資料:


,datetime,datetimeEpoch,tempmax,tempmin,temp,feelslikemax,feelslikemin,feelslike,dew,humidity,...,City,Temp_Range,Heat_Index,Severity_Score,Condition_Code,Month,Season,Day_of_Week,Is_Weekend,Health_Risk_Score
0,2024-09-07,1.725692e+09,106.1,91.0,98.5,104.0,88.1,95.9,51.5,21.0,...,Phoenix,15.1,95.918703,4.4300,NaN,9.0,Fall,Saturday,True,10.522170
1,2024-09-08,1.725779e+09,103.9,87.0,95.4,100.5,84.7,92.3,48.7,21.5,...,Phoenix,16.9,92.281316,3.8800,0.0,9.0,Fall,Sunday,True,10.062332
2,2024-09-09,1.725865e+09,105.0,83.9,94.7,99.9,81.6,90.6,41.7,16.9,...,Phoenix,21.1,90.599165,3.6300,0.0,9.0,Fall,Monday,False,9.673387
3,2024-09-10,1.725952e+09,106.1,81.2,93.9,100.6,79.5,89.8,39.1,15.7,...,Phoenix,24.9,89.638811,2.8512,0.0,9.0,Fall,Tuesday,False,9.411519
4,2024-09-11,1.726038e+09,106.1,82.1,94.0,101.0,80.0,90.0,40.1,15.9,...,Phoenix,24.0,89.760414,3.3908,0.0,9.0,Fall,Wednesday,False,9.515179


欄位型別:
datetime              object
datetimeEpoch        float64
tempmax              float64
tempmin              float64
temp                 float64
feelslikemax         float64
feelslikemin         float64
feelslike            float64
dew                  float64
humidity             float64
precip               float64
precipprob           float64
precipcover          float64
preciptype            object
snow                 float64
snowdepth            float64
windgust             float64
windspeed            float64
winddir              float64
pressure             float64
cloudcover           float64
visibility           float64
solarradiation       float64
solarenergy          float64
uvindex              float64
severerisk           float64
sunrise               object
sunriseEpoch         float64
sunset                object
sunsetEpoch          float64
moonphase            float64
conditions            object
description           object
icon                  object
stations

In [4]:
# 自動偵測空氣品質相關目標欄位
import re
candidate_patterns = re.compile(r"aqi|air|quality|pm", re.IGNORECASE)
columns = rawData.columns
candidates = [c for c in columns if candidate_patterns.search(c)]

# 若無直接 AQI/PM 欄位，使用替代指標
fallback_priority = ["Health_Risk_Score", "Severity_Score", "Condition_Code"]

if len(candidates) == 0:
    for fb in fallback_priority:
        if fb in columns:
            target_col = fb
            reason = f"使用替代指標 {fb} 作為目標，代表健康/環境風險分數。"
            break
    else:
        raise ValueError("找不到可作為空氣品質或健康風險的目標欄位。")
else:
    # 多個候選時，選擇變異數較大的作為代表
    variances = {c: rawData[c].var() for c in candidates if rawData[c].dtype != 'O'}
    if variances:
        target_col = max(variances, key=variances.get)
        reason = f"選取候選欄位中變異數最大的 {target_col} 為目標。"
    else:
        target_col = candidates[0]
        reason = f"選取第一個字串型候選欄位 {target_col} 為目標。"

print("候選欄位:", candidates)
print("最終目標欄位:", target_col)
print("選擇理由:", reason)

候選欄位: ['tempmax', 'tempmin']
最終目標欄位: tempmax
選擇理由: 選取候選欄位中變異數最大的 tempmax 為目標。


In [5]:
# 數值特徵相關性 (Pearson / Spearman)
import numpy as np

numeric_cols = [c for c in rawData.columns if rawData[c].dtype != 'O' and c != target_col]
subset = rawData[numeric_cols + [target_col]].dropna()

pearson = subset[numeric_cols].corrwith(subset[target_col], method='pearson').rename('pearson')
spearman = subset[numeric_cols].corrwith(subset[target_col], method='spearman').rename('spearman')

corr_df = pd.concat([pearson, spearman], axis=1)
# 加入絕對值排序
corr_df['abs_pearson'] = corr_df['pearson'].abs()
corr_df['abs_spearman'] = corr_df['spearman'].abs()

print(f"目標欄位: {target_col}")
print("前 10 名 (依 Pearson 絕對值):")
display(corr_df.sort_values('abs_pearson', ascending=False).head(10))
print("前 10 名 (依 Spearman 絕對值):")
display(corr_df.sort_values('abs_spearman', ascending=False).head(10))

/Users/james_w/NCHU/jupyter_note/.venv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/james_w/NCHU/jupyter_note/.venv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


目標欄位: tempmax
前 10 名 (依 Pearson 絕對值):


/Users/james_w/NCHU/jupyter_note/.venv/lib/python3.13/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


,pearson,spearman,abs_pearson,abs_spearman
feelslikemax,0.976927,0.972307,0.976927,0.972307
temp,0.964727,0.951374,0.964727,0.951374
feelslike,0.948264,0.940425,0.948264,0.940425
tempmin,0.904271,0.863284,0.904271,0.863284
feelslikemin,0.895715,0.861294,0.895715,0.861294
Heat_Index,0.691638,0.566794,0.691638,0.566794
humidity,-0.680360,-0.585399,0.680360,0.585399
pressure,-0.560347,-0.475751,0.560347,0.475751
severerisk,0.324562,0.174663,0.324562,0.174663
solarenergy,0.290708,0.254390,0.290708,0.254390


前 10 名 (依 Spearman 絕對值):


,pearson,spearman,abs_pearson,abs_spearman
feelslikemax,0.976927,0.972307,0.976927,0.972307
temp,0.964727,0.951374,0.964727,0.951374
feelslike,0.948264,0.940425,0.948264,0.940425
tempmin,0.904271,0.863284,0.904271,0.863284
feelslikemin,0.895715,0.861294,0.895715,0.861294
humidity,-0.680360,-0.585399,0.680360,0.585399
Heat_Index,0.691638,0.566794,0.691638,0.566794
pressure,-0.560347,-0.475751,0.560347,0.475751
winddir,-0.259324,-0.289895,0.259324,0.289895
solarenergy,0.290708,0.254390,0.290708,0.254390


In [6]:
# 類別特徵與目標的相關比率 (Correlation Ratio, η)
from collections import defaultdict

def correlation_ratio(categories, values):
    # 移除缺失
    mask = ~pd.isna(categories) & ~pd.isna(values)
    categories = categories[mask]
    values = values[mask]
    if len(values) == 0:
        return np.nan
    groups = defaultdict(list)
    for c, v in zip(categories, values):
        groups[c].append(v)
    total_mean = np.mean(values)
    n_total = len(values)
    ss_between = sum(len(vs) * (np.mean(vs) - total_mean) ** 2 for vs in groups.values())
    ss_total = sum((v - total_mean) ** 2 for v in values)
    if ss_total == 0:
        return 0.0
    return (ss_between / ss_total) ** 0.5

categorical_cols = [c for c in rawData.columns if rawData[c].dtype == 'O' and c != target_col]
eta_scores = {}
for c in categorical_cols:
    try:
        eta_scores[c] = correlation_ratio(rawData[c], rawData[target_col])
    except Exception:
        eta_scores[c] = np.nan

eta_series = pd.Series(eta_scores, name='eta').sort_values(ascending=False)
print("類別特徵與目標 (η) 前 10 名:")
display(eta_series.head(10))

類別特徵與目標 (η) 前 10 名:


stations       0.998437
sunrise        0.994910
sunset         0.989958
City           0.794461
description    0.253781
datetime       0.143868
Day_of_Week    0.117657
conditions     0.082575
icon           0.082575
source         0.041579
Name: eta, dtype: float64

In [7]:
# 彙整最相關特徵 (前 8 名)
# 數值: 取 Pearson 與 Spearman 平均絕對值
numeric_top = corr_df[['abs_pearson','abs_spearman']].mean(axis=1).rename('avg_abs').sort_values(ascending=False).head(8)
cat_top = eta_series.head(8)

summary_df = pd.DataFrame({
    'numeric_feature': numeric_top.index,
    'numeric_score': numeric_top.values
})

summary_df_cat = pd.DataFrame({
    'categorical_feature': cat_top.index,
    'eta_score': cat_top.values
})

print("目標欄位:", target_col)
print("\n數值特徵相關 Top 8 (平均 |Pearson| 與 |Spearman|):")
display(summary_df)
print("\n類別特徵相關 Top 8 (η):")
display(summary_df_cat)

目標欄位: tempmax

數值特徵相關 Top 8 (平均 |Pearson| 與 |Spearman|):


,numeric_feature,numeric_score
0,feelslikemax,0.974617
1,temp,0.958050
2,feelslike,0.944345
3,tempmin,0.883777
4,feelslikemin,0.878504
5,humidity,0.632879
6,Heat_Index,0.629216
7,pressure,0.518049



類別特徵相關 Top 8 (η):


,categorical_feature,eta_score
0,stations,0.998437
1,sunrise,0.994910
2,sunset,0.989958
3,City,0.794461
4,description,0.253781
5,datetime,0.143868
6,Day_of_Week,0.117657
7,conditions,0.082575


## 結果說明與後續建議

請執行上述分析各程式碼區塊後，觀察：
- 數值特徵 (溫度、濕度、風、壓力、雲量等) 與目標變數的相關係數。
- 類別特徵 (城市、季節、天氣狀況、是否週末等) 的 η 值高低。

### 典型可能強相關的影響因子
1. 溫度範圍 (`Temp_Range`) / 體感指數 (`Heat_Index`)：可能與健康風險或空氣品質代理變數相關。
2. 濕度 (`humidity`) 與 露點 (`dew`)：影響污染物擴散與人體感受。
3. 風速 (`windspeed`) / 陣風 (`windgust`)：高風速通常有助於污染物稀釋，相關方向值得檢查正負號。
4. 雲量 (`cloudcover`) / 太陽輻射 (`solarradiation`)：太陽輻射影響光化學反應 (臭氧生成)。
5. 氣壓 (`pressure`)：高壓系統可能伴隨穩定大氣、污染物累積。
6. 季節 (`Season`)：季節性排放與氣象型態差異。

### 後續可加強的方向
- 若取得實際污染物濃度 (PM2.5, NOx, O3) 或 AQI 欄位，可重新跑一次相關分析以提升準確性。
- 可加入時間序列滯後特徵 (前一天風速、溫度等) 探索延遲效應。
- 測試簡單模型 (例如 RandomForest) 做特徵重要度比較，以補充線性相關的限制。

需要我進一步幫你產生視覺化 (例如熱力圖、散佈圖) 或建立預測模型嗎？可以直接告訴我。